# 1. Coastal El Niño / La Niña episodes from ICEN

**What it does**
- Reads Peru's coastal El Niño index (ICEN), monthly from January 2001.
- Flags a month as **El Niño** when ICEN ≥ +0.4 °C and **La Niña** when ICEN ≤ −0.4 °C, if it is part of a run of at least 3 such months.
- Rebuilds the `ENSO`, `Episode` and `Episode_ID` columns (the original file used > +0.5 °C and < −0.7 °C).
- Prints the episode table: 28 episodes in 2001–2026, 23 of them inside the 2001–2022 study period.

**Inputs**
- `data/raw/icen_monthly_original.csv`

**Outputs**
- `data/processed/icen_monthly_pm04.csv`, read by notebooks 5 and 6

Run this notebook from the `notebooks/` folder, top to bottom. It needs no drifter data, so it runs in seconds.

## Relabel the episodes

1. Mark the months above +0.4 °C or below −0.4 °C.
2. Keep only runs of at least 3 consecutive months.
3. Number each run in time order and save the relabelled file.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Coastal El Niño / La Niña rule used everywhere in this notebook:
# El Niño when ICEN >= +0.4 °C, La Niña when ICEN <= -0.4 °C, for at least 3 consecutive months.
# (The episodes in Ashita's icen_monthly.csv used > +0.5 °C and < -0.7 °C.)
ENSO_WARM, ENSO_COLD, ENSO_MIN_MONTHS = 0.4, -0.4, 3

ICEN_SOURCE = Path("../data/raw/icen_monthly_original.csv")
assert ICEN_SOURCE.exists(), ICEN_SOURCE
icen_relabelled = pd.read_csv(ICEN_SOURCE)


def qualifying_runs(active):
    """Months inside a run of at least ENSO_MIN_MONTHS consecutive True values."""
    run_id = (active != active.shift()).cumsum()
    return active & active.groupby(run_id).transform("size").ge(ENSO_MIN_MONTHS)


warm = qualifying_runs(icen_relabelled["ICEN"] >= ENSO_WARM)
cold = qualifying_runs(icen_relabelled["ICEN"] <= ENSO_COLD)
phase = pd.Series(np.select([warm, cold], ["El Niño", "La Niña"], ""), index=icen_relabelled.index)
in_episode = phase.ne("")
episode_start = in_episode & phase.ne(phase.shift())
icen_relabelled["ENSO"] = phase.where(in_episode, "Neutral")
icen_relabelled["Episode"] = phase.where(in_episode, "No")
icen_relabelled["Episode_ID"] = episode_start.cumsum().where(in_episode)

# The rest of the notebook (Ashita's code) reads the relabelled file
ICEN_CSV = Path("../data/processed/icen_monthly_pm04.csv")
icen_relabelled.to_csv(ICEN_CSV, index=False)

episodes_pm04 = (icen_relabelled.dropna(subset=["Episode_ID"]).groupby("Episode_ID")
                 .agg(phase=("Episode", "first"), start=("Date", "first"), end=("Date", "last"),
                      months=("ICEN", "size"), peak_icen=("ICEN", lambda s: s.abs().max())))
print(f"Episodes with the ±{ENSO_WARM} °C, {ENSO_MIN_MONTHS}-month rule: {len(episodes_pm04)} "
      f"({episodes_pm04['phase'].value_counts().to_dict()})")
print(episodes_pm04.to_string())

Episodes with the ±0.4 °C, 3-month rule: 28 ({'La Niña': 15, 'El Niño': 13})
              phase       start         end  months  peak_icen
Episode_ID                                                    
1.0         La Niña  2001-06-01  2002-01-01       8       1.09
2.0         El Niño  2002-03-01  2002-05-01       3       0.81
3.0         El Niño  2002-10-01  2003-01-01       4       0.79
4.0         La Niña  2003-04-01  2003-08-01       5       1.30
5.0         La Niña  2004-04-01  2004-08-01       5       1.11
6.0         La Niña  2005-02-01  2005-12-01      11       1.23
7.0         La Niña  2006-03-01  2006-05-01       3       0.78
8.0         El Niño  2006-08-01  2007-01-01       6       0.94
9.0         La Niña  2007-03-01  2008-01-01      11       1.64
10.0        El Niño  2008-04-01  2008-09-01       6       1.13
11.0        El Niño  2009-05-01  2009-09-01       5       0.75
12.0        La Niña  2010-07-01  2011-03-01       9       1.25
13.0        La Niña  2011-09-01  2011-12-